# EDA2 — Đánh giá cấu hình giao dịch Fashion-MNIST

Notebook này chỉ giữ bằng chứng cần cho các quyết định tiếp theo: độ hợp lệ của bộ giao dịch, đánh đổi theo K, khác biệt giữa BoVW nhị phân và hai cách thêm vị trí, cùng trạng thái khai thác thử.

**Cấu hình release đang dùng:** K=32, seed=123, BoVW nhị phân. Đây là lựa chọn để phân tích, không phải nghiệm tối ưu đã được chứng minh. Các phép so sánh theo lớp là hậu nghiệm và không tham gia chọn cấu hình. Notebook đọc các bảng đã lưu; không tự chạy lại mining hoặc ghi lên artifact nguồn.


## 1. Nạp bảng kết quả đã lưu

In [1]:
using Pkg
using TOML
using Markdown
using Statistics

function find_repo_root(start=pwd())
    current = abspath(start)
    while true
        isfile(joinpath(current, "Project.toml")) && isfile(joinpath(current, "src", "EclatFIM.jl")) && return current
        parent = dirname(current)
        parent == current && error("Không tìm thấy thư mục gốc dự án từ $start")
        current = parent
    end
end

const REPO_ROOT = find_repo_root()
Pkg.activate(REPO_ROOT)
using EclatFIM
const EDA2 = EclatFIM.FashionMNIST.FashionMNISTEDA2
const EDA2_DIR = joinpath(REPO_ROOT, "results", "fashion", "eda2", "v2")
read_eda2(parts...) = begin
    path = joinpath(EDA2_DIR, parts...)
    isfile(path) || error("Thiếu kết quả EDA2: $path")
    last(EDA2._read_csv(path))
end

function show_table(headers, rows)
    escape_cell(value) = replace(replace(string(value), "|" => "\\|"), "\n" => " ")
    lines = ["| " * join(escape_cell.(headers), " | ") * " |",
             "| " * join(fill("---", length(headers)), " | ") * " |"]
    append!(lines, ["| " * join(escape_cell.(row), " | ") * " |" for row in rows])
    display(Markdown.parse(join(lines, "\n")))
end

println("Saved EDA2 results: ", EDA2_DIR)


[ Info: Precompiling Statistics [10745b16-79ce-11e8-11f9-7d13ad32a3b2] (caches not reused: 1 for different source file path)


Precompiling packages...


   0.5 s  ✓ Statistics (serial)
  1 dependency successfully precompiled in 1 seconds
  Activating project at `/run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining`


[ Info: Precompiling EclatFIM [f035e606-8504-4775-8232-c73e28f25aa6] (caches not reused: 3 for different dependency version already loaded)
Precompiling 

packages...


   1.1 s  ✓ Statistics → SparseArraysExt


... (12 outputs left out)

  21.3 s  ✓ ImageCore


   5.5 s  ✓ PNGFiles


Saved EDA2 results: /run/media/fernir/Data/Documents/HCMUS/SEMESTER7/DataMining/TeamProject-FrequentItemsetMining/results/fashion/eda2/v2

   2.5 s  ✓ EclatFIM
  15 dependencies successfully precompiled in 46 seconds. 76 already precompiled.
  1 dependency precompiled but a different version is currently loaded (Statistics). Restart julia to access the new version. Otherwise, 15 dependents of this package may trigger further precompilation to work with the unexpected version.


## 2. Đầu vào hợp lệ chưa?

In [2]:
manifest_rows = read_eda2("input_manifest.csv")
validation = TOML.parsefile(joinpath(EDA2_DIR, "validation_report.toml"))
@assert length(manifest_rows) == 45
@assert all(row["status"] == "PASS" for row in manifest_rows)
@assert all(parse(Int, row["record_count"]) == 70_000 for row in manifest_rows)
@assert sort(unique(parse(Int, row["K"]) for row in manifest_rows)) == [16, 32, 64, 128, 256]
@assert sort(unique(parse(Int, row["seed"]) for row in manifest_rows)) == [42, 123, 2026]
@assert sort(unique(row["variant"] for row in manifest_rows)) == ["S0", "S1", "S3"]

show_table(["Phạm vi kiểm tra", "Kết quả"], [
    ["Bộ giao dịch", "45/45 PASS"],
    ["Số ảnh trong mỗi bộ", "70.000"],
    ["Cấu hình", "5 giá trị K × 3 seed × 3 biểu diễn"],
])


|    Phạm vi kiểm tra |                            Kết quả |
| -------------------:| ----------------------------------:|
|        Bộ giao dịch |                         45/45 PASS |
| Số ảnh trong mỗi bộ |                             70.000 |
|            Cấu hình | 5 giá trị K × 3 seed × 3 biểu diễn |


## 3. Đánh đổi theo số cụm K

Bảng đối chiếu độ méo codebook với độ dài trung bình của giao dịch BoVW nhị phân và số cặp thường xuyên trong pilot Eclat tại minsup 3%, giới hạn độ dài hai mục. Số cặp là chỉ báo sàng lọc của lượt pilot, không phải thước đo chất lượng tổng quát.


In [3]:
codebooks = read_eda2("codebook", "codebook_metrics_by_k.csv")
transactions = read_eda2("transactions", "transaction_metrics_45.csv")
pilots = read_eda2("pilot_eclat", "pilot_common_45.csv")
tradeoff_rows = Vector{Vector{Any}}()
for codebook in sort(codebooks; by=row -> parse(Int, row["K"]))
    k = parse(Int, codebook["K"])
    tx = filter(row -> parse(Int, row["K"]) == k && row["variant"] == "S0", transactions)
    pilot = filter(row -> parse(Int, row["K"]) == k && row["variant"] == "S0", pilots)
    push!(tradeoff_rows, [k,
        round(parse(Float64, codebook["distortion_mean"]); digits=4),
        round(mean(parse(Float64, row["mean_length"]) for row in tx); digits=2),
        round(mean(parse(Float64, row["frequent_2_count"]) for row in pilot); digits=1),
        "3 seed"])
end
show_table(["K", "Độ méo TB", "Độ dài TB, BoVW nhị phân", "Cặp phổ biến TB ở pilot", "Lặp"], tradeoff_rows)


|   K | Độ méo TB | Độ dài TB, BoVW nhị phân | Cặp phổ biến TB ở pilot |    Lặp |
| ---:| ---------:| ------------------------:| -----------------------:| ------:|
|  16 |    0.4451 |                     8.86 |                   120.0 | 3 seed |
|  32 |    0.3457 |                    12.28 |                   479.0 | 3 seed |
|  64 |    0.2715 |                    15.13 |                  1471.3 | 3 seed |
| 128 |    0.2183 |                    17.96 |                  1545.0 | 3 seed |
| 256 |    0.1771 |                    20.37 |                   661.0 | 3 seed |


K tăng làm giảm độ méo codebook nhưng làm giao dịch dài hơn. K=32 được giữ làm điểm cân bằng cho lần phân tích này; K=16 tạo giao dịch ngắn hơn nhưng độ méo cao hơn, còn K=64 giảm độ méo thêm nhưng giao dịch dài hơn. K=32 không được chứng minh là tối ưu.


## 4. BoVW nhị phân và hai biểu diễn có vị trí tại K=32, seed=123

In [4]:
tx_selected = filter(row -> parse(Int, row["K"]) == 32 && parse(Int, row["seed"]) == 123, transactions)
pilot_selected = filter(row -> parse(Int, row["K"]) == 32 && parse(Int, row["seed"]) == 123, pilots)
encoding_rows = Vector{Vector{Any}}()
for variant in ("S0", "S1", "S3")
    tx = only(filter(row -> row["variant"] == variant, tx_selected))
    pilot = only(filter(row -> row["variant"] == variant, pilot_selected))
    label = variant == "S0" ? "BoVW nhị phân" : variant == "S1" ? "BoVW + vùng 2×2" : "BoVW + dải trên/giữa/dưới"
    push!(encoding_rows, [label,
        round(parse(Float64, tx["mean_length"]); digits=2),
        round(100 * parse(Float64, tx["unique_ratio"]); digits=2),
        parse(Int, pilot["frequent_1_count"]),
        parse(Int, pilot["frequent_2_count"]),
        pilot["status"]])
end
show_table(["Biểu diễn", "Độ dài TB", "Giao dịch phân biệt (%)", "Mục đơn ở 3%", "Cặp ở 3%", "Trạng thái pilot"], encoding_rows)


|                 Biểu diễn | Độ dài TB | Giao dịch phân biệt (%) | Mục đơn ở 3% | Cặp ở 3% |        Trạng thái pilot |
| -------------------------:| ---------:| -----------------------:| ------------:| --------:| -----------------------:|
|             BoVW nhị phân |     12.27 |                   92.37 |           32 |      482 | COMPLETED*LENGTH*CAPPED |
|           BoVW + vùng 2×2 |      18.2 |                   99.13 |          102 |     1905 | COMPLETED*LENGTH*CAPPED |
| BoVW + dải trên/giữa/dưới |     16.48 |                   99.38 |           84 |     1658 | COMPLETED*LENGTH*CAPPED |


S1/S3 giữ thêm thông tin vùng hoặc dải nên có không gian mục lớn hơn; vì vậy không so sánh trực tiếp số lượng tập mục giữa các biểu diễn như thể cùng một thước đo chất lượng. BoVW nhị phân được dùng trong báo cáo để tập trung vào đồng xuất hiện từ thị giác, không lưu vị trí trong ảnh.


## 5. Khai thác sâu và cổng kết luận

In [5]:
shortlist = read_eda2("pilot_eclat", "shortlist_decision.csv")
deep = read_eda2("pilot_eclat", "pilot_deep_runs.csv")
review = read_eda2("interpretability", "pattern_interpretability.csv")
selected_codebooks = count(row -> lowercase(row["selected"]) == "true", shortlist)
status_counts = Dict(status => count(row -> row["completion_status"] == status, deep)
                     for status in unique(row["completion_status"] for row in deep))
pending_visual = count(row -> row["visual_coherence"] == "pending", review)
pending_redundancy = count(row -> row["redundancy"] == "pending", review)
status_summary = join(["$status=$n" for (status, n) in sort(collect(status_counts))], ", ")

show_table(["Kiểm tra", "Kết quả"], [
    ["Codebook trong shortlist", "$selected_codebooks/$(length(shortlist))"],
    ["Lượt mining sâu", "$(length(deep)) điều kiện; trạng thái: $status_summary"],
    ["Mẫu chờ duyệt trực quan", "$pending_visual/$(length(review))"],
    ["Mẫu chờ rà trùng lặp", "$pending_redundancy/$(length(review))"],
    ["Bộ kết quả tối đại/đóng chính xác", "Chưa xác định đầy đủ do giới hạn độ dài khai thác"],
])


|                          Kiểm tra |                                                Kết quả |
| ---------------------------------:| ------------------------------------------------------:|
|          Codebook trong shortlist |                                                   9/15 |
|                   Lượt mining sâu | 108 điều kiện; trạng thái: COMPLETED*LENGTH*CAPPED=108 |
|           Mẫu chờ duyệt trực quan |                                                319/319 |
|              Mẫu chờ rà trùng lặp |                                                319/319 |
| Bộ kết quả tối đại/đóng chính xác |      Chưa xác định đầy đủ do giới hạn độ dài khai thác |


## 6. Kết luận cho bước tiếp theo

- Cấu hình dùng trong báo cáo: **K=32, seed=123, BoVW nhị phân**, trên bộ 70.000 ảnh đã kiểm tra.
- EDA2 chưa chứng minh đây là cấu hình tốt nhất. Trạng thái chọn cấu hình vẫn **INCONCLUSIVE** vì phần đánh giá trực quan và mức dư thừa của các mẫu trong shortlist chưa hoàn tất.
- Các số đo theo lớp chỉ dùng để mô tả hậu nghiệm; chúng không chứng minh khả năng phân loại.
- Nếu làm thí nghiệm phân loại, phải tách train/test trước; mọi bước phụ thuộc dữ liệu—ngưỡng lọc, codebook, khai thác tập mục và chọn đặc trưng—chỉ fit trên train rồi áp dụng cố định cho test.

Các phép tính chi tiết nằm trong `src/applications/Fashion-Mnist/transaction_eda/`; notebook này chỉ tổng hợp đầu ra đã lưu. Không có cell nào tự chạy lại Eclat hoặc ghi đè kết quả.
